# 📖 Navegador de Capítulos — Estrategias No Convencionales

Explora los 33 capítulos del libro organizados en 8 bloques temáticos.

Este notebook te permite:
- 🔍 Descubrir todas las estrategias no convencionales
- 🔍 Filtrar por bloque temático
- 📖 Leer capítulos completos en formato Markdown
- 🔎 Buscar términos específicos en todos los capítulos
-  Ver estadísticas del proyecto

## Celda 1: Imports y configuración

In [1]:
# Celda 1: Imports, detección de entorno y clonación automática en Colab
from pathlib import Path
from urllib.parse import quote
import os
import subprocess
import json
from IPython.display import display, HTML, Markdown

# =============================================================================
# CONFIGURACIÓN GLOBAL
# =============================================================================
GITHUB_USER = "akitxu"
GITHUB_REPO = "Estrategias_No_Convencionales"
GITHUB_BRANCH = "main"
GITHUB_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

# =============================================================================
# DETECCIÓN DE ENTORNO
# =============================================================================
def esta_en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IN_COLAB = esta_en_colab()

print("🔍 Detectando entorno y preparando proyecto...")
print("=" * 70)

if IN_COLAB:
    print("✅ Entorno: Google Colab")
else:
    print("✅ Entorno: Local (Jupyter)")

# =============================================================================
# PREPARACIÓN DEL PROYECTO
# =============================================================================
def encontrar_raiz_local():
    """
    Busca la raíz del repositorio empezando en el directorio actual
    y recorriendo sus padres.
    """
    actual = Path.cwd().resolve()

    for candidato in [actual, *actual.parents]:
        if (
            (candidato / "notebooks").is_dir()
            or (candidato / "chapters").is_dir()
            or (candidato / ".git").exists()
        ):
            return candidato

    return actual


def preparar_proyecto():
    """
    Devuelve la raíz del proyecto tanto en local como en Google Colab.
    """
    if IN_COLAB:
        raiz = Path("/content") / GITHUB_REPO

        if not raiz.exists():
            print(f"📥 Clonando repositorio desde GitHub...")
            print(f"   URL: {GITHUB_URL}")

            resultado = subprocess.run(
                [
                    "git",
                    "clone",
                    "--depth",
                    "1",
                    "--branch",
                    GITHUB_BRANCH,
                    GITHUB_URL,
                    str(raiz),
                ],
                capture_output=True,
                text=True,
            )

            if resultado.returncode != 0:
                raise RuntimeError(
                    "No se pudo clonar el repositorio:\n"
                    f"{resultado.stderr}"
                )

            print(f"✅ Repositorio clonado exitosamente en: {raiz}")

        elif not (raiz / ".git").exists():
            raise RuntimeError(
                f"La ruta {raiz} existe, pero no parece ser un repositorio Git."
            )
        else:
            print(f"✅ Repositorio encontrado en: {raiz}")

        return raiz

    # Local
    raiz = encontrar_raiz_local()
    print(f"✅ Usando directorio local: {raiz.absolute()}")
    return raiz


PROJECT_ROOT = preparar_proyecto()

# =============================================================================
# CONFIGURAR RUTAS
# =============================================================================
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
CHAPTERS_DIR = PROJECT_ROOT / "chapters"
CORE_DIR = PROJECT_ROOT / "core"

print(f"\n📂 PROJECT_ROOT: {PROJECT_ROOT}")
print(f"📓 Notebooks: {NOTEBOOKS_DIR}")
print(f"📚 Capítulos: {CHAPTERS_DIR}")
print(f"🧠 Core: {CORE_DIR}")
print(f"\n✅ Notebooks existe: {NOTEBOOKS_DIR.is_dir()}")
print(f"✅ Capítulos existe: {CHAPTERS_DIR.is_dir()}")
print(f"✅ Core existe: {CORE_DIR.is_dir()}")
print("=" * 70)

🔍 Detectando entorno y preparando proyecto...
✅ Entorno: Local (Jupyter)
✅ Usando directorio local: /home/enri/Estrategias_No_Convencionales

📂 PROJECT_ROOT: /home/enri/Estrategias_No_Convencionales
📓 Notebooks: /home/enri/Estrategias_No_Convencionales/notebooks
📚 Capítulos: /home/enri/Estrategias_No_Convencionales/chapters
🧠 Core: /home/enri/Estrategias_No_Convencionales/core

✅ Notebooks existe: True
✅ Capítulos existe: True
✅ Core existe: True


## Celda 2: Función para listar capítulos (8 bloques, 33 capítulos)

In [2]:
# Celda 2: Función para listar capítulos (bloques temáticos)
import re


def clasificar_capitulo(archivo):
    """
    Clasifica un archivo .md en un bloque temático según su nombre.
    """
    nombre = archivo.stem
    nombre_lower = nombre.lower()

    # Prefacio / índices / portada
    if any(
        palabra in nombre_lower
        for palabra in ("prefacio", "indice", "índice", "portada")
    ):
        return "Prefacio"

    # Apéndices / glosario
    if any(
        palabra in nombre_lower
        for palabra in ("apendice", "apéndice", "glosario")
    ):
        return "Apéndices"

    # Extraer número inicial
    coincidencia = re.match(r"^(\d+)", nombre)

    if not coincidencia:
        return "Sin clasificar"

    numero = int(coincidencia.group(1))

    if 1 <= numero <= 4:
        return "Bloque I - Comportamiento y Psicología"
    if 5 <= numero <= 7:
        return "Bloque II - Cuantitativas y Arbitraje"
    if 8 <= numero <= 10:
        return "Bloque III - Estrategias Basadas en Eventos"
    if 11 <= numero <= 13:
        return "Bloque IV - Macro y Globales"
    if 14 <= numero <= 16:
        return "Bloque V - Alternativas y Exóticas"
    if 17 <= numero <= 20:
        return "Bloque VI - Nicho y Tácticas"
    if 21 <= numero <= 23:
        return "Bloque VII - Alto Riesgo"
    if numero >= 24:
        return "Bloque VIII - Indicadores Técnicos"

    return "Sin clasificar"


def obtener_capitulos():
    """
    Obtiene todos los archivos .md y los organiza por bloques temáticos.
    """
    if not CHAPTERS_DIR.is_dir():
        print(f"⚠️ No se encontró el directorio: {CHAPTERS_DIR}")
        return {}

    bloques = {}

    for archivo in sorted(CHAPTERS_DIR.rglob("*.md")):
        if archivo.name.startswith("."):
            continue

        bloque = clasificar_capitulo(archivo)
        bloques.setdefault(bloque, []).append(archivo)

    return bloques


capitulos_organizados = obtener_capitulos()

total_capitulos = sum(
    len(archivos)
    for archivos in capitulos_organizados.values()
)

print(f"✅ Se encontraron {total_capitulos} capítulos organizados en {len(capitulos_organizados)} bloques\n")

for bloque, archivos in capitulos_organizados.items():
    print(f"   • {bloque}: {len(archivos)} archivos")

✅ Se encontraron 16 capítulos organizados en 4 bloques

   • Prefacio: 4 archivos
   • Bloque I - Comportamiento y Psicología: 7 archivos
   • Bloque II - Cuantitativas y Arbitraje: 4 archivos
   • Apéndices: 1 archivos


## Celda 3: Menú interactivo visual

In [12]:
# Celda 3: Menú visual de capítulos con enlaces Local y GitHub

from html import escape
from urllib.parse import quote
from IPython.display import display, HTML

print("📖 NAVEGADOR DE CAPÍTULOS")
print("=" * 70)
print("El botón 'Leer local' abre el archivo Markdown servido por Jupyter.\n")


def url_local_jupyter(ruta_relativa):
    """
    Construye una URL para abrir un archivo dentro del servidor Jupyter local.

    Ejemplo de salida:
    /files/chapters/05_Cap5_Pairs_Trading.md
    """
    ruta_url = quote(ruta_relativa.as_posix(), safe="/")

    # JupyterLab sirve archivos estáticos bajo /files/
    return f"/files/{ruta_url}"


def construir_enlaces_capitulo(archivo):
    """
    Construye las URLs local y GitHub para un capítulo Markdown.
    """
    ruta_relativa = archivo.relative_to(PROJECT_ROOT)
    ruta_url = quote(ruta_relativa.as_posix(), safe="/")

    local_url = url_local_jupyter(ruta_relativa)

    github_url = (
        f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}"
        f"/blob/{GITHUB_BRANCH}/{ruta_url}"
    )

    return local_url, github_url


def mostrar_menu_capitulos():
    """
    Muestra las tarjetas de navegación de capítulos.

    - Leer local abre el archivo Markdown desde el servidor Jupyter.
    - Ver en GitHub abre el archivo renderizado por GitHub.
    """
    if not capitulos_organizados:
        print("⚠️ No hay capítulos disponibles para mostrar.")
        print("   Ejecuta primero la celda de escaneo de capítulos.")
        return

    colores_bloque = {
        "Prefacio": "#6c757d",
        "Bloque I - Comportamiento y Psicología": "#007bff",
        "Bloque II - Cuantitativas y Arbitraje": "#6f42c1",
        "Bloque III - Estrategias Basadas en Eventos": "#20c997",
        "Bloque IV - Macro y Globales": "#fd7e14",
        "Bloque V - Alternativas y Exóticas": "#e83e8c",
        "Bloque VI - Nicho y Tácticas": "#17a2b8",
        "Bloque VII - Alto Riesgo": "#dc3545",
        "Bloque VIII - Indicadores Técnicos": "#28a745",
        "Apéndices": "#6c757d",
        "Sin clasificar": "#adb5bd",
    }

    html = """
    <style>
        .cap-card {
            background: linear-gradient(135deg, #f8f9fa 0%, #e9ecef 100%);
            color: #333333;
            padding: 15px 20px;
            margin: 12px 0;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0, 0, 0, 0.10);
            transition: transform 0.2s, box-shadow 0.2s;
            border-left: 6px solid #007bff;
            display: flex;
            justify-content: space-between;
            align-items: center;
            flex-wrap: wrap;
            gap: 12px;
        }

        .cap-card:hover {
            transform: translateY(-2px);
            box-shadow: 0 4px 8px rgba(0, 0, 0, 0.15);
        }

        .cap-info {
            flex-grow: 1;
            min-width: 220px;
        }

        .cap-titulo {
            font-size: 1.15em;
            font-weight: bold;
            margin-bottom: 5px;
            color: #2c3e50;
        }

        .cap-archivo {
            font-size: 0.85em;
            color: #6c757d;
            font-family: monospace;
            overflow-wrap: anywhere;
        }

        .btn-group {
            display: flex;
            gap: 8px;
            flex-wrap: wrap;
        }

        .btn {
            display: inline-block;
            padding: 8px 14px;
            border-radius: 5px;
            text-decoration: none !important;
            font-weight: bold;
            font-size: 0.85em;
            border: 1px solid transparent;
            cursor: pointer;
            white-space: nowrap;
            transition: background 0.2s, color 0.2s;
        }

        .btn-local {
            background: #f37626;
            color: #ffffff !important;
            border-color: #f37626;
        }

        .btn-local:hover {
            background: #d9651f;
            color: #ffffff !important;
        }

        .btn-github {
            background: #e9dff7;
            color: #3f2860 !important;
            border-color: #d2bde9;
        }

        .btn-github:hover {
            background: #f4edfb;
            color: #2d1c48 !important;
            border-color: #bda0da;
        }

        h2 {
            color: #2c3e50;
            border-bottom: 3px solid #007bff;
            padding-bottom: 10px;
        }

        h3 {
            color: #495057;
            margin-top: 25px;
            border-left: 4px solid #007bff;
            padding-left: 10px;
        }
    </style>

    <h2>📖 Navegador de Capítulos</h2>
    <p style="color: #666; font-size: 0.95em;">
        “Leer local” abre el archivo Markdown en una pestaña nueva desde Jupyter.
        “GitHub” abre el capítulo con el visor web de GitHub.
    </p>
    """

    for bloque, archivos in capitulos_organizados.items():
        color = colores_bloque.get(bloque, "#007bff")

        html += (
            f"<h3 style='border-left-color: {color}; color: {color};'>"
            f"{escape(bloque)} ({len(archivos)})"
            f"</h3>"
        )

        for archivo in archivos:
            local_url, github_url = construir_enlaces_capitulo(archivo)

            nombre_limpio = (
                archivo.stem
                .replace("_", " ")
                .replace("Cap", "Capítulo ")
            )

            html += f"""
            <div class="cap-card" style="border-left-color: {color};">
                <div class="cap-info">
                    <div class="cap-titulo">
                        📘 {escape(nombre_limpio)}
                    </div>
                    <div class="cap-archivo">
                        {escape(archivo.name)}
                    </div>
                </div>

                <div class="btn-group">
                    <a
                        href="{escape(local_url, quote=True)}"
                        target="_blank"
                        class="btn btn-local"
                        title="Abrir este capítulo desde el servidor Jupyter"
                    >
                        📖 Leer local
                    </a>

                    <a
                        href="{escape(github_url, quote=True)}"
                        target="_blank"
                        class="btn btn-github"
                        title="Abrir este capítulo en GitHub"
                    >
                        🐙 Ver en GitHub
                    </a>
                </div>
            </div>
            """

    display(HTML(html))


mostrar_menu_capitulos()

📖 NAVEGADOR DE CAPÍTULOS
El botón 'Leer local' abre el archivo Markdown servido por Jupyter.



## Celda 4: Visualizador de capítulos

In [5]:
 # Celda 4: Funciones de visualización de capítulos
def buscar_archivo_capitulo(numero_capitulo):
    """
    Busca un archivo de capítulo por su número (ej. 1, 01, 001...).
    """
    texto_numero = str(numero_capitulo).zfill(2)

    for archivos in capitulos_organizados.values():
        for archivo in archivos:
            if archivo.stem.startswith(texto_numero):
                return archivo

    return None


def ver_capitulo(numero_capitulo, max_lineas=30):
    """
    Visualiza las primeras `max_lineas` líneas de un capítulo.
    """
    archivo = buscar_archivo_capitulo(numero_capitulo)

    if archivo is None:
        print(f"❌ No se encontró el capítulo {numero_capitulo}.")
        return

    contenido = archivo.read_text(encoding="utf-8")
    lineas = contenido.splitlines()

    print(f"📖 Visualizando: {archivo.name}")
    print("=" * 70)
    print("\n".join(lineas[:max_lineas]))

    if len(lineas) > max_lineas:
        print("\n... contenido truncado ...")


def leer_capitulo_completo(numero_capitulo):
    """
    Lee y muestra el contenido completo de un capítulo en formato Markdown.
    """
    archivo = buscar_archivo_capitulo(numero_capitulo)

    if archivo is None:
        print(f"❌ No se encontró el capítulo {numero_capitulo}.")
        return

    contenido = archivo.read_text(encoding="utf-8")
    display(Markdown(contenido))


print("✅ Funciones de visualización disponibles:")
print("   • ver_capitulo(5) → Vista previa del Capítulo 5")
print("   • leer_capitulo_completo(7) → Capítulo 7 completo en Markdown")

✅ Funciones de visualización disponibles:
   • ver_capitulo(5) → Vista previa del Capítulo 5
   • leer_capitulo_completo(7) → Capítulo 7 completo en Markdown


## Celda 5: Buscador de contenido

In [6]:
# Celda 5: Buscador de contenido en los capítulos
import pandas as pd


def buscar_en_capitulos(termino):
    """
    Busca un término en todos los capítulos y muestra los resultados.
    """
    print(f"🔍 Buscando: '{termino}'")
    print("=" * 70)

    resultados = []

    for bloque, archivos in capitulos_organizados.items():
        for archivo in archivos:
            try:
                contenido = archivo.read_text(encoding="utf-8").lower()
            except Exception:
                continue

            if termino.lower() in contenido:
                ocurrencias = contenido.count(termino.lower())
                resultados.append((archivo.stem, bloque, ocurrencias))

    if resultados:
        print(f"\n✅ Se encontraron {len(resultados)} capítulos con '{termino}':\n")

        df_resultados = pd.DataFrame(
            resultados,
            columns=["Capítulo", "Bloque", "Ocurrencias"]
        )
        df_resultados = df_resultados.sort_values(
            "Ocurrencias", ascending=False
        ).reset_index(drop=True)

        display(df_resultados)
    else:
        print(f"\n❌ No se encontraron resultados para '{termino}'")


print("💡 Uso: buscar_en_capitulos('Sharpe')")

💡 Uso: buscar_en_capitulos('Sharpe')


## Celda 6: Estadísticas del proyecto

In [7]:
# Celda 6: Estadísticas del proyecto
def estadisticas_proyecto():
    """
    Muestra estadísticas del proyecto (capítulos, líneas, palabras).
    """
    total_capitulos = sum(
        len(archivos)
        for archivos in capitulos_organizados.values()
    )
    total_lineas = 0
    total_palabras = 0

    for archivos in capitulos_organizados.values():
        for archivo in archivos:
            try:
                contenido = archivo.read_text(encoding="utf-8")
            except Exception:
                continue

            total_lineas += len(contenido.splitlines())
            total_palabras += len(contenido.split())

    print("📊 ESTADÍSTICAS DEL PROYECTO")
    print("=" * 70)
    print(f"📚 Total de capítulos: {total_capitulos}")
    print(f"📝 Total de líneas: {total_lineas:,}")
    print(f"📖 Total de palabras: {total_palabras:,}")
    print(f"📂 Directorio: {CHAPTERS_DIR}")
    print("=" * 70)


estadisticas_proyecto()

📊 ESTADÍSTICAS DEL PROYECTO
📚 Total de capítulos: 16
📝 Total de líneas: 3,349
📖 Total de palabras: 45,967
📂 Directorio: /home/enri/Estrategias_No_Convencionales/chapters


## Celda 7: Guía de uso

In [8]:
print("""
📖 GUÍA DE USO RÁPIDA
=====================

Funciones disponibles:

1. mostrar_menu_capitulos()
   → Muestra el menú visual interactivo con todos los capítulos

2. ver_capitulo(n)
   → Visualiza las primeras 30 líneas del capítulo n
   Ejemplo: ver_capitulo(2)

3. leer_capitulo_completo(n)
   → Lee el capítulo completo en formato Markdown
   Ejemplo: leer_capitulo_completo(5)

4. buscar_en_capitulos('termino')
   → Busca un término en todos los capítulos
   Ejemplo: buscar_en_capitulos('Piotroski')

5. estadisticas_proyecto()
   → Muestra estadísticas del libro (capítulos, líneas, palabras)

EJEMPLOS DE USO:
    mostrar_menu_capitulos()
    ver_capitulo(2)
    leer_capitulo_completo(5)
    buscar_en_capitulos('Sharpe')
    estadisticas_proyecto()
""")


📖 GUÍA DE USO RÁPIDA

Funciones disponibles:

1. mostrar_menu_capitulos()
   → Muestra el menú visual interactivo con todos los capítulos

2. ver_capitulo(n)
   → Visualiza las primeras 30 líneas del capítulo n
   Ejemplo: ver_capitulo(2)

3. leer_capitulo_completo(n)
   → Lee el capítulo completo en formato Markdown
   Ejemplo: leer_capitulo_completo(5)

4. buscar_en_capitulos('termino')
   → Busca un término en todos los capítulos
   Ejemplo: buscar_en_capitulos('Piotroski')

5. estadisticas_proyecto()
   → Muestra estadísticas del libro (capítulos, líneas, palabras)

EJEMPLOS DE USO:
    mostrar_menu_capitulos()
    ver_capitulo(2)
    leer_capitulo_completo(5)
    buscar_en_capitulos('Sharpe')
    estadisticas_proyecto()

